# SQL Analysis - Online News Popularity

## Objective
Use SQL to analyse article popularity patterns and identify factors linked with higher social media shares.

In [1]:
# Import Libraries
import pandas as pd
import sqlite3

In [2]:
# Load the dataset
df = pd.read_csv("../data/processed/cleaned_online_news.csv")

df.head()

,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,average_token_length,...,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares,log_shares
0,12.0,219.0,0.663594,1.0,0.815385,4.0,2.0,1.0,0.0,4.680365,...,0.7,-0.350000,-0.600,-0.200000,0.500000,-0.187500,0.000000,0.187500,593,6.386879
1,9.0,255.0,0.604743,1.0,0.791946,3.0,1.0,1.0,0.0,4.913725,...,0.7,-0.118750,-0.125,-0.100000,0.000000,0.000000,0.500000,0.000000,711,6.568078
2,9.0,211.0,0.575130,1.0,0.663866,3.0,1.0,1.0,0.0,4.393365,...,1.0,-0.466667,-0.800,-0.133333,0.000000,0.000000,0.500000,0.000000,1500,7.313887
3,9.0,531.0,0.503788,1.0,0.665635,9.0,0.0,1.0,0.0,4.404896,...,0.8,-0.369697,-0.600,-0.166667,0.000000,0.000000,0.500000,0.000000,1200,7.090910
4,13.0,1072.0,0.415646,1.0,0.540890,19.0,19.0,20.0,0.0,4.682836,...,1.0,-0.220192,-0.500,-0.050000,0.454545,0.136364,0.045455,0.136364,505,6.226537


In [3]:
# Create SQLite database
conn = sqlite3.connect("../data/processed/news_popularity.db")

df.to_sql("news", conn, index=False, if_exists="replace")

39644

In [4]:
# Check table columns
query = """
SELECT *
FROM news
LIMIT 5;
"""

pd.read_sql(query, conn)

,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,average_token_length,...,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares,log_shares
0,12.0,219.0,0.663594,1.0,0.815385,4.0,2.0,1.0,0.0,4.680365,...,0.7,-0.350000,-0.600,-0.200000,0.500000,-0.187500,0.000000,0.187500,593,6.386879
1,9.0,255.0,0.604743,1.0,0.791946,3.0,1.0,1.0,0.0,4.913725,...,0.7,-0.118750,-0.125,-0.100000,0.000000,0.000000,0.500000,0.000000,711,6.568078
2,9.0,211.0,0.575130,1.0,0.663866,3.0,1.0,1.0,0.0,4.393365,...,1.0,-0.466667,-0.800,-0.133333,0.000000,0.000000,0.500000,0.000000,1500,7.313887
3,9.0,531.0,0.503788,1.0,0.665635,9.0,0.0,1.0,0.0,4.404896,...,0.8,-0.369697,-0.600,-0.166667,0.000000,0.000000,0.500000,0.000000,1200,7.090910
4,13.0,1072.0,0.415646,1.0,0.540890,19.0,19.0,20.0,0.0,4.682836,...,1.0,-0.220192,-0.500,-0.050000,0.454545,0.136364,0.045455,0.136364,505,6.226537


In [ ]:
# Basic dataset overview
query = """
SELECT
    COUNT(*) AS total_articles,
    ROUND(AVG(shares), 2) AS average_shares,
    MIN(shares) AS minimum_shares,
    MAX(shares) AS maximum_shares
FROM news;
"""

pd.read_sql(query, conn)

,total_articles,average_shares,minimum_shares,maximum_shares
0,39644,3395.38,1,843300


In [6]:
# Most popular articles categories
query = """
SELECT
    CASE
        WHEN data_channel_is_lifestyle = 1 THEN 'Lifestyle'
        WHEN data_channel_is_entertainment = 1 THEN 'Entertainment'
        WHEN data_channel_is_bus = 1 THEN 'Business'
        WHEN data_channel_is_socmed = 1 THEN 'Social Media'
        WHEN data_channel_is_tech = 1 THEN 'Technology'
        WHEN data_channel_is_world = 1 THEN 'World'
        ELSE 'Other'
    END AS category,
    COUNT(*) AS total_articles,
    ROUND(AVG(shares), 2) AS average_shares
FROM news
GROUP BY category
ORDER BY average_shares DESC;
"""

pd.read_sql(query, conn)

,category,total_articles,average_shares
0,Other,6134,5945.19
1,Lifestyle,2099,3682.12
2,Social Media,2323,3629.38
3,Technology,7346,3072.28
4,Business,6258,3063.02
5,Entertainment,7057,2970.49
6,World,8427,2287.73


In [7]:
# Weekday popularity analysis
query = """
SELECT
    CASE
        WHEN weekday_is_monday = 1 THEN 'Monday'
        WHEN weekday_is_tuesday = 1 THEN 'Tuesday'
        WHEN weekday_is_wednesday = 1 THEN 'Wednesday'
        WHEN weekday_is_thursday = 1 THEN 'Thursday'
        WHEN weekday_is_friday = 1 THEN 'Friday'
        WHEN weekday_is_saturday = 1 THEN 'Saturday'
        WHEN weekday_is_sunday = 1 THEN 'Sunday'
    END AS weekday,
    COUNT(*) AS total_articles,
    ROUND(AVG(shares), 2) AS average_shares
FROM news
GROUP BY weekday
ORDER BY average_shares DESC;
"""

pd.read_sql(query, conn)

,weekday,total_articles,average_shares
0,Saturday,2453,4078.19
1,Sunday,2737,3746.74
2,Monday,6661,3647.03
3,Wednesday,7435,3303.41
4,Friday,5701,3285.18
5,Tuesday,7390,3202.50
6,Thursday,7267,3178.60


In [8]:
# Relationship between number of images and shares
query = """
SELECT
    CASE
        WHEN num_imgs = 0 THEN 'No images'
        WHEN num_imgs BETWEEN 1 AND 3 THEN '1-3 images'
        WHEN num_imgs BETWEEN 4 AND 7 THEN '4-7 images'
        ELSE '8+ images'
    END AS image_group,
    COUNT(*) AS total_articles,
    ROUND(AVG(shares), 2) AS average_shares
FROM news
GROUP BY image_group
ORDER BY average_shares DESC;
"""

pd.read_sql(query, conn)

,image_group,total_articles,average_shares
0,8+ images,8212,4696.09
1,No images,6987,4395.78
2,4-7 images,2138,3365.13
3,1-3 images,22307,2606.10


In [9]:
# Relationship between number of videos and shares
query = """
SELECT
    CASE
        WHEN num_videos = 0 THEN 'No videos'
        WHEN num_videos BETWEEN 1 AND 2 THEN '1-2 videos'
        WHEN num_videos BETWEEN 3 AND 5 THEN '3-5 videos'
        ELSE '6+ videos'
    END AS video_group,
    COUNT(*) AS total_articles,
    ROUND(AVG(shares), 2) AS average_shares
FROM news
GROUP BY video_group
ORDER BY average_shares DESC;
"""

pd.read_sql(query, conn)

,video_group,total_articles,average_shares
0,6+ videos,2025,4550.92
1,1-2 videos,11672,4267.14
2,3-5 videos,921,3499.33
3,No videos,25026,2891.47
